In [ ]:
#@title Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Weather forecasting and storm risk nowcasting with `Earth Engine`, BigQuery ML, and Gemini

| | |
| --- | --- |
| **Author** | [Google Cloud People and Planet AI](https://github.com/GoogleCloudPlatform/python-docs-samples) |
| **GitHub** | [GoogleCloudPlatform/python-docs-samples](https://github.com/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb) |
| **Open in Colab** | [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb) |
| **Open in Colab Enterprise** | [![Open in Colab Enterprise](https://cloud.google.com/static/colab/images/icon-colab-enterprise.svg)](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fpython-docs-samples%2Ffeat%2Fmodernize-ppai-weather-forecasting%2Fpeople-and-planet-ai%2Fweather-forecasting%2FREADME.ipynb) |
| **Open in Vertex AI Workbench** | [![Open in Vertex AI Workbench](https://gstatic.com/cloudssh/images/open-btn.svg)](https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/python-docs-samples/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb) |

## Overview

Short-term precipitation nowcasting and flood-risk assessment combine satellite observations, terrain topography, and foundation forecasting models to anticipate severe weather impacts hours before peak runoff occurs.

In this tutorial, you walk through an end-to-end weather forecasting and severe-storm analysis workflow on Google Cloud:

1. **Multi-band satellite and terrain retrieval (Google Earth Engine)**: Fetch calibrated half-hourly precipitation rates from **NASA GPM IMERG V07** (`NASA/GPM_L3/IMERG_V07`), geostationary cloud and moisture channels from **NOAA GOES-19** (`NOAA/GOES/19/MCMIPF`) or **GOES-16** (`NOAA/GOES/16/MCMIPF`), and surface elevation from **MERIT DEM** (`MERIT/DEM/v1_0_3`).
2. **Hourly storm time-series ingestion (Apache Beam / Dataflow & BigQuery)**: Run an Apache Beam pipeline (`load_weather_time_series.py`) to load hourly storm progression observations into a BigQuery table.
3. **Zero-shot foundation time-series forecasting (BigQuery ML `AI.FORECAST`)**: Forecast upcoming hourly rainfall rates and 90% prediction intervals in SQL using Google Research's **TimesFM 2.5** model without training a custom neural network from scratch.
4. **Multimodal storm and orographic flood-risk synthesis (`gemini-3.8-flash`)**: Combine the 3-panel satellite and terrain composite image with the `AI.FORECAST` precipitation trajectory using the Google Gen AI SDK and Pydantic structured outputs (`analyze_storm_risk.py`).
5. **Global medium-range ensemble forecasting with Google DeepMind WeatherNext**: Explore how to query **WeatherNext** (`weathernext_2_0_0` and `weathernext_3_0_0_0p1deg`) 64-member probabilistic forecasts in Earth Engine and BigQuery.

### Storm case studies included

* **Hurricane Melissa (`2025-10-28T17:00:00Z`, Jamaica landfall — default)**: Category 5 landfall in southwestern Jamaica (`18.1N, 78.0W`), where coastal rainbands interact with the steep Blue Mountains (`2,256 m` peak elevation) and where Google DeepMind's WeatherNext models demonstrated 5-day advance rapid-intensification skill.
* **Hurricane Helene (`2024-09-27T06:00:00Z`, Southern Appalachians)**: Historic inland atmospheric moisture plume interacting with Blue Ridge mountain topography (`35.60N, 82.55W`, up to `2,037 m` elevation) to drive extreme orographic rainfall amplification.

### Objectives

* Retrieve and visualize multi-band Earth Engine satellite (`GOES-19`/`GOES-16`), precipitation (`GPM IMERG V07`), and elevation (`MERIT DEM`) rasters.
* Populate a BigQuery time-series table using Apache Beam (`load_weather_time_series.py`).
* Execute zero-shot precipitation nowcasting in BigQuery ML with `AI.FORECAST` and `TimesFM 2.5` (`forecast_precipitation.py`).
* Generate a validated `StormRiskAssessment` report from multimodal satellite imagery and tabular forecasts using `gemini-3.8-flash` (`analyze_storm_risk.py`).

### Costs

This tutorial uses billable components of Google Cloud:

* [BigQuery](https://cloud.google.com/bigquery/pricing) (storage and `AI.FORECAST` query processing)
* [Dataflow](https://cloud.google.com/dataflow/pricing) (if running the Apache Beam pipeline on `DataflowRunner`; defaults to local `DirectRunner` in this notebook)
* [Vertex AI / Gemini API](https://cloud.google.com/vertex-ai/generative-ai/pricing) (`gemini-3.8-flash` multimodal token usage)
* [Cloud Storage](https://cloud.google.com/storage/pricing) (temporary staging bucket for BigQuery file loads)

## Get started

### Clone the sample repository and install dependencies

Fetch the `people-and-planet-ai/weather-forecasting` sample scripts and install the required Python packages into the current notebook kernel (~30 seconds).

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/GoogleCloudPlatform/python-docs-samples.git"
BRANCH = "feat/modernize-ppai-weather-forecasting"
SAMPLE_DIR = "people-and-planet-ai/weather-forecasting"
REPO_DIR = "ppai-weather-forecasting"

if not os.path.exists("fetch_weather_patch.py"):
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1", REPO_URL, REPO_DIR],
            check=True,
        )
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "init", "--cone"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", SAMPLE_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "-c", "advice.detachedHead=false", "checkout", "-f", "FETCH_HEAD"],
        check=True,
    )
    os.chdir(os.path.join(REPO_DIR, SAMPLE_DIR))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

Install the required libraries for Earth Engine, Apache Beam, BigQuery, Gemini (`google-genai`), and Plotly visualization.

In [ ]:
%pip install -q --upgrade "apache-beam[gcp]>=2.60.0" "earthengine-api>=1.0.0" "google-cloud-bigquery>=3.25.0" "google-genai>=1.0.0" "numpy>=1.26.0" "pandas>=2.0.0" "pillow>=10.0.0" "plotly>=5.20.0" "pydantic>=2.7.0"

import importlib
importlib.invalidate_caches()

### Authenticate your notebook environment

If you are running this notebook in Google Colab, authenticate with your Google Cloud user credentials.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Configure your Google Cloud project and storm preset

Set your `PROJECT_ID`, `BUCKET_NAME`, `BIGQUERY_DATASET`, and choose between `melissa_2025` (Hurricane Melissa, Oct 28, 2025, Jamaica) and `helene_2024` (Hurricane Helene, Sept 27, 2024, Southern Appalachians).

In [ ]:
import logging
import os
import subprocess
import warnings

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module=r"pandas\.core\.arrays\.masked")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning:pandas.core.arrays.masked"

PROJECT_ID = "example-project-id"  # @param {type:"string"}
LOCATION = "US"  # @param {type:"string"}
BUCKET_NAME = "example-weather-bucket"  # @param {type:"string"}
BIGQUERY_DATASET = "example_weather_forecasting"  # @param {type:"string"}
BIGQUERY_TABLE = "example_storm_observations"  # @param {type:"string"}
STORM_PRESET = "melissa_2025"  # @param ["melissa_2025", "helene_2024"]

if not PROJECT_ID or PROJECT_ID == "example-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", LOCATION)
if not BUCKET_NAME or BUCKET_NAME == "example-weather-bucket":
    BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-weather-forecasting")
BIGQUERY_DATASET = os.environ.get("BIGQUERY_DATASET", BIGQUERY_DATASET)
BIGQUERY_TABLE = os.environ.get("BIGQUERY_TABLE", BIGQUERY_TABLE)
STORM_PRESET = os.environ.get("STORM_PRESET", STORM_PRESET)

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print(f"Project ID:       {PROJECT_ID}")
print(f"Staging Bucket:   gs://{BUCKET_NAME}")
print(f"BigQuery Dataset: {PROJECT_ID}.{BIGQUERY_DATASET}")
print(f"Storm Preset:     {STORM_PRESET}")

### Enable required Google Cloud APIs and create the staging bucket

Enable the BigQuery, Vertex AI (`aiplatform`), Earth Engine, and Dataflow APIs, and create a regional Cloud Storage bucket with soft-delete disabled for temporary staging files (~30 seconds).

In [ ]:
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com dataflow.googleapis.com earthengine.googleapis.com storage.googleapis.com --project={PROJECT_ID}

bucket_check = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if bucket_check.returncode != 0:
    !gcloud storage buckets create gs://{BUCKET_NAME} --project={PROJECT_ID} --location={LOCATION} --soft-delete-duration=0s
else:
    !gcloud storage buckets update gs://{BUCKET_NAME} --clear-soft-delete

## 1. Retrieve and visualize satellite, precipitation, and terrain patches

In [`fetch_weather_patch.py`](fetch_weather_patch.py), you combine three geospatial layers around the storm center:

* **Geostationary cloud & moisture (`NOAA/GOES/19/MCMIPF` or `NOAA/GOES/16/MCMIPF`)**: Captures cloud-top brightness temperatures and atmospheric moisture transport (`CMI_C01`, `CMI_C02`, `CMI_C03`, `CMI_C13`). Note that **GOES-19** became the operational GOES-East satellite on April 7, 2025.
* **Calibrated precipitation (`NASA/GPM_L3/IMERG_V07`)**: Provides half-hourly merged microwave-infrared precipitation rates (`precipitation` band in `mm/hr`).
* **Terrain elevation (`MERIT/DEM/v1_0_3`)**: Provides surface elevation (`elevation` in meters), which is critical for identifying **orographic precipitation enhancement** where moist cyclonic flow is forced up steep mountain slopes.

Run the cell below to fetch the multi-band storm patch and render the interactive 3-panel figure.

In [ ]:
import fetch_weather_patch
import visualize

patch_data = fetch_weather_patch.get_storm_patch_arrays(
    project_id=PROJECT_ID,
    storm_preset=STORM_PRESET,
    patch_size=64,
)
print(f"Storm:       {patch_data['storm_name']}")
print(f"Timestamp:   {patch_data['iso_time']}")
print(f"Satellite:   {patch_data['goes_satellite']}")
print(f"Data Source: {patch_data['source']}")

overview_fig = visualize.create_storm_overview_figure(patch_data)
overview_fig.show()

Save the 3-panel composite as a PNG image (`storm_composite.png`) so you can pass it as a multimodal visual input to `gemini-3.8-flash` in Step 4.

In [ ]:
COMPOSITE_IMAGE_PATH = "storm_composite.png"
png_bytes = visualize.export_composite_png_bytes(patch_data)
with open(COMPOSITE_IMAGE_PATH, "wb") as f:
    f.write(png_bytes)
print(f"Saved {COMPOSITE_IMAGE_PATH} ({len(png_bytes):,} bytes)")

## 2. Load hourly storm observations into BigQuery with `Apache Beam`

Next, execute [`load_weather_time_series.py`](load_weather_time_series.py) (`[START dataflow_weather_load_time_series]`). This Apache Beam pipeline constructs a 36-hour sequence of hourly storm observations (`observation_time`, `storm_name`, `precipitation_mm_hr`, `cloud_moisture_index`, and `elevation_m`) and writes them into BigQuery using `WriteToBigQuery` (~15–25 seconds on `DirectRunner`).

In [ ]:
# Expected duration: ~15-25 seconds on DirectRunner
!python load_weather_time_series.py \
  --project={PROJECT_ID} \
  --dataset={BIGQUERY_DATASET} \
  --table={BIGQUERY_TABLE} \
  --runner=DirectRunner \
  --temp_location=gs://{BUCKET_NAME}/temp

## 3. Zero-shot precipitation nowcasting with BigQuery ML `AI.FORECAST` (`TimesFM 2.5`)

Instead of extracting thousands of `.npz` patches and training a custom convolutional neural network from scratch, BigQuery ML provides zero-shot time-series forecasting out of the box via **`AI.FORECAST`** powered by Google Research's **TimesFM 2.5** foundation model.

Run [`forecast_precipitation.py`](forecast_precipitation.py) (`[START bigquery_weather_forecast_precipitation]`) to forecast the next 6 hours of precipitation rates alongside 90% prediction intervals (~10–20 seconds).

In [ ]:
# Expected duration: ~10-20 seconds
!python forecast_precipitation.py {PROJECT_ID} {BIGQUERY_DATASET} {BIGQUERY_TABLE}

### Visualize observed `GPM IMERG V07` rainfall versus `AI.FORECAST` nowcast

Query the historical series and the `AI.FORECAST` predictions from BigQuery and plot them together with the 90% confidence band.

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{BIGQUERY_DATASET}.{BIGQUERY_TABLE}"

hist_sql = f"SELECT observation_time, precipitation_mm_hr FROM `{table_ref}` ORDER BY observation_time ASC"
fc_sql = f"""
    SELECT
      forecast_timestamp,
      forecast_value AS forecasted_precipitation_mm_hr,
      prediction_interval_lower_bound AS lower_bound_mm_hr,
      prediction_interval_upper_bound AS upper_bound_mm_hr
    FROM
      AI.FORECAST(
        TABLE `{table_ref}`,
        data_col => 'precipitation_mm_hr',
        timestamp_col => 'observation_time',
        model => 'TimesFM 2.5',
        id_cols => ['storm_name'],
        horizon => 6,
        confidence_level => 0.90
      )
    ORDER BY forecast_timestamp ASC
"""
historical_rows = [dict(r) for r in bq_client.query(hist_sql).result()]
forecast_rows = [dict(r) for r in bq_client.query(fc_sql).result()]

forecast_fig = visualize.create_forecast_comparison_figure(
    historical_rows=historical_rows,
    forecast_rows=forecast_rows,
    storm_name="Hurricane Melissa",
)
forecast_fig.show()

## 4. Multimodal storm and orographic flood-risk synthesis with Gemini 3.8 Flash

Quantitative rainfall forecasts become much more actionable when combined with spatial cloud structure and terrain topography. In [`analyze_storm_risk.py`](analyze_storm_risk.py) (`[START googlegenaisdk_weather_analyze_storm_risk]`), you call **`gemini-3.8-flash`** on the `global` endpoint (`aiplatform.googleapis.com`) using the Google Gen AI SDK (`google-genai`).

The model receives:
1. The 3-panel satellite and DEM composite PNG (`types.Part.from_bytes`), and
2. The `AI.FORECAST` hourly trajectory,

and returns a strictly typed `StormRiskAssessment` Pydantic object.

In [ ]:
!python analyze_storm_risk.py {PROJECT_ID} {COMPOSITE_IMAGE_PATH}

## 5. Customize for local current weather forecasting (`San Francisco Bay Area`)

You can extend this exact pipeline to live local weather forecasting for any region. By default, the cell below targets the **San Francisco Bay Area (`37.7749°N, -122.4194°W`)** and automatically:

1. Selects **GOES-18 West (`NOAA/GOES/18/MCMIPF`)** for Western longitudes (`longitude < -105.0°`) or **GOES-19 East (`NOAA/GOES/19/MCMIPF`)** for Eastern longitudes to capture the latest 10-minute cloud and marine-layer scan.
2. Pairs the satellite imagery with **MERIT DEM (`MERIT/DEM/v1_0_3`)** topography (from `-2.6 m` across San Francisco Bay to `851.7 m` across Mount Tamalpais and the Santa Cruz Mountains).
3. Queries the latest operational **NOAA GFS (`NOAA/GFS0P25`)** 72-hour forecast trajectory (`+6h` to `+72h`) in Earth Engine.

Customize `LOCAL_REGION_NAME`, `LOCAL_LATITUDE`, and `LOCAL_LONGITUDE` below to test the San Francisco Bay Area or your own coordinates.

In [ ]:
LOCAL_REGION_NAME = "San Francisco Bay Area, CA"  # @param {type:"string"}
LOCAL_LATITUDE = 37.7749  # @param {type:"number"}
LOCAL_LONGITUDE = -122.4194  # @param {type:"number"}
LOCAL_REGION_NAME = os.environ.get("LOCAL_REGION_NAME", LOCAL_REGION_NAME)

local_result = fetch_weather_patch.get_local_weather_forecast(
    project_id=PROJECT_ID,
    region_name=LOCAL_REGION_NAME,
    latitude=float(LOCAL_LATITUDE),
    longitude=float(LOCAL_LONGITUDE),
)
local_patch = local_result["patch_data"]
print(f"Region:      {local_patch['storm_name']}")
print(f"Satellite:   {local_patch['goes_satellite']}")
print(f"Data Source: {local_patch['source']}")
for row in local_result["forecast_rows"]:
    print(
        f"  +{row['lead_hours']:02d}h ({row['forecast_time_utc']}): "
        f"{row['temp_c']:.1f}°C ({row['temp_f']:.1f}°F), "
        f"RH {row['relative_humidity_pct']:.0f}%, wind {row['wind_speed_ms']:.1f} m/s, "
        f"clouds {row['cloud_cover_pct']:.0f}%, precip {row['precip_mm']:.2f} mm"
    )

local_fig = visualize.create_storm_overview_figure(local_patch)
local_fig.show()

### Synthesize a localized microclimate and weather risk report with Gemini 3.8 Flash

Save the 3-panel local composite image (`local_weather_composite.png`) and pass it alongside the 72-hour `NOAA/GFS0P25` forecast trajectory into [`analyze_storm_risk.py`](analyze_storm_risk.py) (`gemini-3.8-flash`).

In [ ]:
LOCAL_COMPOSITE_PATH = "local_weather_composite.png"
with open(LOCAL_COMPOSITE_PATH, "wb") as f:
    f.write(visualize.export_composite_png_bytes(local_patch))

os.environ["WEATHER_FORECAST_CONTEXT"] = local_result["forecast_context"]
!python analyze_storm_risk.py {PROJECT_ID} {LOCAL_COMPOSITE_PATH}

## 6. Scaling to global medium-range ensemble forecasts with `WeatherNext`

While `NASA/GPM_L3/IMERG_V07` + `AI.FORECAST` provides rapid time-series nowcasting from historical satellite observations, **Google DeepMind WeatherNext** delivers full 3D global medium-range ensemble weather predictions across Google Earth Engine, BigQuery (via Analytics Hub), and Cloud Storage (Zarr v3):

* **WeatherNext 3 (`projects/gcp-public-data-weathernext/assets/weathernext_3_0_0_0p1deg` and `weathernext_3_0_0_0p05deg`)**: Initialized every hour (24 runs/day) with a 64-member ensemble, directly ingesting geostationary satellite mosaics and providing 1-hour accumulated precipitation (`total_precipitation_1hr`), 2m temperature (`temperature_2m`), and 10m/100m wind across precomputed ensemble distribution percentiles (`mean`, `p10`, `p25`, `p50`, `p75`, `p90`).
* **WeatherNext 2 (`projects/gcp-public-data-weathernext/assets/weathernext_2_0_0`)**: 64-member Functional Generative Network (FGN) ensemble forecasts at 0.25-degree resolution.

> **Note on Access**: WeatherNext datasets on Earth Engine and BigQuery are part of the Google Cloud Publisher Catalog. After your Google account is approved via the [WeatherNext Data Request Form](https://developers.google.com/weathernext/guides/models), you can query ensemble precipitation percentiles directly in Earth Engine:

```python
weathernext3 = (
    ee.ImageCollection("projects/gcp-public-data-weathernext/assets/weathernext_3_0_0_0p1deg")
    .filterDate("2025-10-28T00:00:00Z", "2025-10-29T00:00:00Z")
    .select([
        "total_precipitation_1hr_mean",
        "total_precipitation_1hr_p50",
        "total_precipitation_1hr_p90",
    ])
)
```

## Cleaning up

To avoid incurring ongoing charges to your Google Cloud account for the resources created in this tutorial, delete the BigQuery dataset and temporary Cloud Storage staging files.

In [ ]:
!bq rm -r -f -d {PROJECT_ID}:{BIGQUERY_DATASET}
!gcloud storage rm --recursive gs://{BUCKET_NAME}/temp